# Fermi-GBM Time-Resolved Spectral Analysis Pipeline
Each cell corresponds to one pipeline stage — run them top to bottom, or re-run any single stage independently.

## Cell 1 — Imports & Setup

In [22]:
import os
import logging
import numpy as np
import matplotlib
matplotlib.use('Agg')   # swap to 'inline' if running interactively
import matplotlib.pyplot as plt
from astropy.table import Table
import astropy.units as u

from threeML import (
    silence_logs, download_GBM_trigger_data, TimeSeriesBuilder,
    OGIPLike, DataList, Model, PointSource, BayesianAnalysis, plot_spectra,
)
from threeML.catalogs import FermiGBMBurstCatalog as FermiGBMBurst
from astromodels import Cutoff_powerlaw, Band, Log_uniform_prior, Truncated_gaussian           # add model name here after adding cell for new model

silence_logs()
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('GBM_pipeline')
print('Imports OK')


Imports OK


## Cell 2 — Configuration

In [23]:
# ── Edit this block only ────────────────────────────────────────────
CONFIG = dict(
    trigger_name   = 'bn140304557',   # GBM trigger ID  (no 'GRB' prefix)
    grb_name       = 'GRB140304557',  # catalog key      (with 'GRB' prefix)
    ra             = 30.653,           # RA  of the burst [deg]  — update!
    dec            =  33.480,           # Dec of the burst [deg]  — update!

    # detector selection
    max_rov        = 60.0,            # keep detectors with ROV < this [deg]

    # Bayesian-blocks settings
    bb_t_start     = -10.0,           # start of BB window [s relative to trigger]
    bb_t_stop      =  50.0,           # end   of BB window
    bb_p0          =  0.01,           # false-alarm probability for BB
    min_bin_width  =  0.05,           # seconds; narrower bins are merged away

    # spectral energy ranges
    nai_range      = '8-33, 37-900',  # keV  (gap skips K-edge ~34-36 keV)
    bgo_range      = '200-40000',     # keV

    # ── Add new model priors here ────────────────────────────────────────────────────────────
    
    # CutoffPL-function priors
    priors_cutoffpl = dict(
        K     = dict(lower=1e-5, upper=1e1),
        index = dict(lower=-2.0, upper=1.0,  mu=-1.3,  sigma=0.5),
        xc    = dict(lower=10.0, upper=1000, mu=300.0, sigma=200.0)
    ),

    # Band-function priors
    priors_band = dict(
        K     = dict(lower=1e-4, upper=1e1),
        alpha = dict(lower=-1.7, upper=1.0,  mu=-1.0,  sigma=0.5),
        xp    = dict(lower=10.0, upper=1000, mu=300.0, sigma=200.0),
        beta  = dict(lower=-4.0, upper=-1.8, mu=-2.5,  sigma=1.0),
    ),

    # ──────────────────────────────────────────────────────────────────────────────────────────
    
    # MultiNest settings
    n_live_points  = 400,             # increase to >=1000 for publication quality

    # output
    outdir         = 'gbm_pipeline_output',
)

# ── Create common output directories ──────────────────────────────────
COMMON_PATHS = {
    'root': CONFIG['outdir'],                 # For common .h5 detector files
    'lc':   os.path.join(CONFIG['outdir'], 'lightcurves'), # For shared lightcurves
}

for d in COMMON_PATHS.values():
    os.makedirs(d, exist_ok=True)

# ── Helper to dynamically create model subdirectories ─────────────────
def get_model_paths(model_name):
    """Creates and returns specific paths for a given spectral model."""
    m_dir = os.path.join(COMMON_PATHS['root'], model_name)
    m_paths = {
        'root':    m_dir,
        'spectra': os.path.join(m_dir, 'spectra'),
        'corners': os.path.join(m_dir, 'corners'),
    }
    for d in m_paths.values():
        os.makedirs(d, exist_ok=True)
    return m_paths

print('Config loaded. Common output directories created at:', CONFIG['outdir'])

Config loaded. Common output directories created at: gbm_pipeline_output


## Cell 3 — Helper Function Definitions

In [24]:
def pick_reference_detector(detectors):
    """Select the first NaI detector as the BB reference, fallback to first overall."""
    nai = [d for d in detectors if d.startswith('n')]
    return nai[0] if nai else detectors[0]


def build_cutoffpl_model(ra, dec, priors_cutoffpl):
    """Return a fresh Band-function Model with the configured priors."""
    band  = Cutoff_powerlaw()
    model = Model(PointSource('GBM', ra, dec, spectral_shape=band))
    p = priors_cutoffpl
    model.GBM.spectrum.main.Cutoff_powerlaw.K.prior = Log_uniform_prior(
        lower_bound=p['K']['lower'], upper_bound=p['K']['upper'])
    model.GBM.spectrum.main.Cutoff_powerlaw.index.prior = Truncated_gaussian(
        lower_bound=p['index']['lower'], upper_bound=p['index']['upper'],
        mu=p['index']['mu'], sigma=p['index']['sigma'])
    model.GBM.spectrum.main.Cutoff_powerlaw.xc.prior = Truncated_gaussian(
        lower_bound=p['xc']['lower'], upper_bound=p['xc']['upper'],
        mu=p['xc']['mu'], sigma=p['xc']['sigma'])
    return model

def build_band_model(ra, dec, priors_band):
    """Return a fresh Band-function Model with the configured priors."""
    band  = Band()
    model = Model(PointSource('GBM', ra, dec, spectral_shape=band))
    p = priors_band
    model.GBM.spectrum.main.Band.K.prior = Log_uniform_prior(
        lower_bound=p['K']['lower'], upper_bound=p['K']['upper'])
    model.GBM.spectrum.main.Band.alpha.bounds = (
        p['alpha']['lower'], p['alpha']['upper'])
    model.GBM.spectrum.main.Band.alpha.prior = Truncated_gaussian(
        lower_bound=p['alpha']['lower'], upper_bound=p['alpha']['upper'],
        mu=p['alpha']['mu'], sigma=p['alpha']['sigma'])
    model.GBM.spectrum.main.Band.xp.prior = Truncated_gaussian(
        lower_bound=p['xp']['lower'], upper_bound=p['xp']['upper'],
        mu=p['xp']['mu'], sigma=p['xp']['sigma'])
    model.GBM.spectrum.main.Band.beta.prior = Truncated_gaussian(
        lower_bound=p['beta']['lower'], upper_bound=p['beta']['upper'],
        mu=p['beta']['mu'], sigma=p['beta']['sigma'])
    return model


print('Helper functions defined.')


Helper functions defined.


## Cell 4 — Detector Selection (ROV < 60°)

In [25]:
log.info('Querying Fermi-GBM catalog for %s ...', CONFIG['grb_name'])

cat = FermiGBMBurst()
cat.query_sources(CONFIG['grb_name'])
grb_info = cat.get_detector_information()[CONFIG['grb_name']]

# ── Fetch catalog source / background intervals ───────────────────────
source_interval     = grb_info['source']['fluence']
background_interval = grb_info['background']['full']
log.info('Source interval    : %s', source_interval)
log.info('Background interval: %s', background_interval)

# ── Manually Specify Detectors ────────────────────────────────────────
# Valid NaI detectors: 'n0' through 'n9', 'na', 'nb'
# Valid BGO detectors: 'b0', 'b1'
detectors = ['n3', 'n4', 'b0']  # <-- Change these to your preferred detectors

log.info('Manually selected detectors done: %s', detectors)


# ── Quick sanity check ────────────────────────────────────────────────
assert len(detectors) > 0, 'No detectors selected. Please add them to the list.'
assert any(d.startswith('n') for d in detectors), 'Need at least one NaI detector for the fit.'


14:27:50  INFO      Querying Fermi-GBM catalog for GRB140304557 ...
14:27:58  INFO      Source interval    : -5.376000-25.856000
14:27:58  INFO      Background interval: -40.830000--11.250000,41.670000-97.920000
14:27:58  INFO      Manually selected detectors done: ['n3', 'n4', 'b0']


## Cell 5 — Download GBM Data

In [26]:
log.info('Downloading TTE / CSPEC / RSP for: %s', detectors)
dload = download_GBM_trigger_data(CONFIG['trigger_name'], detectors=detectors)

# ── Verify all expected file types are present ────────────────────────
for det in detectors:
    for ftype in ('tte', 'cspec', 'rsp'):
        assert ftype in dload[det], f'Missing {ftype} file for detector {det}'

print('Download complete. Files available for:', list(dload.keys()))


14:27:58  INFO      Downloading TTE / CSPEC / RSP for: ['n3', 'n4', 'b0']


Download complete. Files available for: ['n3', 'n4', 'b0']


## Cell 6 — Build Time Series & Fit Backgrounds

In [27]:
time_series = {}

for det in detectors:
    log.info('Processing detector %s ...', det)

    # ── Step 6a: fit background polynomial on CSPEC ───────────────────
    ts_cspec = TimeSeriesBuilder.from_gbm_cspec_or_ctime(
        det,
        cspec_or_ctime_file = dload[det]['cspec'],
        rsp_file            = dload[det]['rsp'],
    )
    bkg_segs = [s.strip() for s in background_interval.split(',')]
    ts_cspec.set_background_interval(*bkg_segs)

    bkg_file = os.path.join(COMMON_PATHS['root'], f'{det}_bkg.h5')
    ts_cspec.save_background(bkg_file, overwrite=True)
    log.info('  Background saved to %s', bkg_file)

    # ── Step 6b: load TTE with restored background ────────────────────
    ts_tte = TimeSeriesBuilder.from_gbm_tte(
        det,
        tte_file           = dload[det]['tte'],
        rsp_file           = dload[det]['rsp'],
        restore_background = bkg_file,
    )
    ts_tte.set_active_time_interval(source_interval)
    time_series[det] = ts_tte
    log.info('  TTE loaded and active interval set.')

print('Time series built for:', list(time_series.keys()))

# ── Quick sanity: view lightcurve of first NaI ────────────────────────
first_nai = next(d for d in detectors if d.startswith('n'))
fig = time_series[first_nai].view_lightcurve(-40, 100)

fig.savefig(os.path.join(COMMON_PATHS['lc'], f'{first_nai}_raw_lc.png'), dpi=100, bbox_inches='tight')
plt.close('all')
print(f'Raw lightcurve saved for {first_nai}')


14:28:11  INFO      Processing detector n3 ...

WARNING RuntimeWarning: invalid value encountered in divide



Loading PHAII Spectra:   0%|                                                                                | …


WARNING RuntimeWarning: invalid value encountered in divide



Finding best polynomial Order:   0%|                                                                          …

Fitting GBM_NAI_03 background:   0%|                                                                         |…

...


WARNING RuntimeWarning: invalid value encountered in divide



Finding best polynomial Order:   0%|                                                                          …

Fitting GBM_BGO_00 background:   0%|                                                                         |…

14:29:31  INFO        Background saved to gbm_pipeline_output/b0_bkg.h5
14:29:35  INFO        TTE loaded and active interval set.


Time series built for: ['n3', 'n4', 'b0']
Raw lightcurve saved for n3


## Cell 7 — Bayesian-Block Time Binning

In [28]:
ref_det = pick_reference_detector(detectors)
ts_ref  = time_series[ref_det]
log.info('Reference detector for BB: %s', ref_det)

# ── Run Bayesian blocks ────────────────────────────────────────────────
ts_ref.create_time_bins(
    CONFIG['bb_t_start'], CONFIG['bb_t_stop'],
    method='bayesblocks', use_background=True, p0=CONFIG['bb_p0'],
)

print('Raw BB bins:')
print('  starts:', ts_ref.bins.starts)
print('  stops :', ts_ref.bins.stops)
print('  widths:', ts_ref.bins.widths)

fig_raw = ts_ref.view_lightcurve(use_binner=True)
fig_raw.savefig(os.path.join(COMMON_PATHS['lc'], f'{ref_det}_bb_raw.png'), dpi=120, bbox_inches='tight')
plt.close('all')


14:29:36  INFO      Reference detector for BB: n3


Raw BB bins:
  starts: [-9.999761998653412, -6.61941796541214, 7.057295024394989, 11.157336056232452]
  stops : [-6.61941796541214, 7.057295024394989, 11.157336056232452, 49.99896001815796]
  widths: [ 3.38034403 13.67671299  4.10004103 38.84162396]


## Cell 8 — Merge Sub-Threshold Bins

In [29]:
min_w = CONFIG['min_bin_width']

bad_bins = {i for i, w in enumerate(ts_ref.bins.widths) if w < min_w}
print(f'Bins narrower than {min_w} s (to be merged): {sorted(bad_bins)}')

edges = [ts_ref.bins.starts[0]]
for i, b in enumerate(ts_ref.bins):
    if i not in bad_bins:
        edges.append(b.stop)

bb_starts = np.array(edges[:-1])
bb_stops  = np.array(edges[1:])
n_bins    = len(bb_starts)

ts_ref.create_time_bins(bb_starts, bb_stops, method='custom')

print(f'Clean bins ({n_bins} total):')
print('  starts:', bb_starts)
print('  stops :', bb_stops)
print('  widths:', bb_stops - bb_starts)

fig_clean = ts_ref.view_lightcurve(use_binner=True)
fig_clean.savefig(os.path.join(COMMON_PATHS['lc'], f'{ref_det}_bb_clean.png'), dpi=120, bbox_inches='tight')
plt.close('all')

assert n_bins > 0, 'Zero bins after merging — check bb_t_start / bb_t_stop or p0'
print(f'\nReady to fit {n_bins} time bin(s).')


Bins narrower than 0.05 s (to be merged): []
Clean bins (4 total):
  starts: [-9.999762   -6.61941797  7.05729502 11.15733606]
  stops : [-6.61941797  7.05729502 11.15733606 49.99896002]
  widths: [ 3.38034403 13.67671299  4.10004103 38.84162396]

Ready to fit 4 time bin(s).


## Cell 9 — Sync Detectors & Build Time-Resolved Plugins

In [30]:
time_resolved_plugins = {}   # {det: [plugin_bin_0, plugin_bin_1, ...]}

for det in detectors:
    ts = time_series[det]
    ts.read_bins(ts_ref)                         # sync to reference bins
    plugin_list = ts.to_spectrumlike(from_bins=True)
    time_resolved_plugins[det] = plugin_list
    log.info('%s  ->  %d time-bin plugins', det, len(plugin_list))

# ── Sanity: all detectors should have the same number of bins ─────────
bin_counts = {d: len(v) for d, v in time_resolved_plugins.items()}
print('Plugin counts per detector:', bin_counts)
assert len(set(bin_counts.values())) == 1, 'Bin count mismatch across detectors!'
print('All detectors in sync. Ready for spectral fitting.')


Creating plugins:   0%|                                                                                       …

14:29:54  INFO      n3  ->  4 time-bin plugins


Creating plugins:   0%|                                                                                       …

14:29:55  INFO      n4  ->  4 time-bin plugins


Creating plugins:   0%|                                                                                       …

14:29:57  INFO      b0  ->  4 time-bin plugins


Plugin counts per detector: {'n3': 4, 'n4': 4, 'b0': 4}
All detectors in sync. Ready for spectral fitting.


## Cell 10 — Time-Resolved Spectral Fitting (CutoffPLModel)

In [31]:
MODEL_PATHS = get_model_paths('cutoffpl')                             # Change this for each new model

PARAM_NAMES = [
    'B_K',     'B_K_n_err',     'B_K_p_err',     'B_K_avg_err',
    'B_alpha', 'B_alpha_n_err', 'B_alpha_p_err',  'B_alpha_avg_err',
    'E_c',  'E_c_n_err',  'E_c_p_err',   'E_c_avg_err',
    'flux',    'flux_low',      'flux_hi'
]

fit_rows = []

for i in range(n_bins):
    log.info('==== Time bin %d / %d ====', i + 1, n_bins)

    # ── Assemble per-detector plugins for this bin ────────────────────
    det_plugins = []
    for det in detectors:
        p = time_resolved_plugins[det][i]
        if det.startswith('b'):
            p.set_active_measurements(CONFIG['bgo_range'])
        else:
            ranges = [r.strip() for r in CONFIG['nai_range'].split(',')]
            p.set_active_measurements(*ranges)
        det_plugins.append(p)

    data_list = DataList(*det_plugins)

    # ── Fresh Band model per bin ──────────────────────────────────────
    model = build_cutoffpl_model(CONFIG['ra'], CONFIG['dec'], CONFIG['priors_cutoffpl'])

    # ── Bayesian fit with MultiNest ───────────────────────────────────
    bayes = BayesianAnalysis(model, data_list)
    bayes.set_sampler('multinest')
    bayes.sampler.setup(n_live_points=CONFIG['n_live_points'])
    bayes.sample()

    # ── νFν spectrum plot ─────────────────────────────────────────────
    fig_spec = plot_spectra(
        bayes.results,
        ene_min=1.0, ene_max=1e5, num_ene=200,
        contour_colors='white',
        contour_style_kwargs=dict(alpha=0.2),
        use_components=True, components_to_use=['total'],
        flux_unit='keV2/(cm2 s keV)',
    )
    ax = plt.gca()
    ax.set_xlabel('Energy (keV)', fontsize=16)
    ax.set_ylabel(r'$\nu\,F_\nu\;[\mathrm{keV\,cm^{-2}\,s^{-1}}]$', fontsize=16)
    ax.set_title(f"{CONFIG['trigger_name']}  bin {i+1}/{n_bins}  "
                 f"[{bb_starts[i]:.3f}, {bb_stops[i]:.3f}] s", fontsize=13)
    fig_spec.savefig(
        os.path.join(MODEL_PATHS['spectra'], f'cutoffpl_spec_bin{i+1:03d}.png'),
        dpi=120, bbox_inches='tight')
    plt.close('all')

    # ── Corner plot ───────────────────────────────────────────────────
    fig_corner = bayes.results.corner_plot()
    fig_corner.savefig(
        os.path.join(MODEL_PATHS['corners'], f'cutoffpl_corner_bin{i+1:03d}.png'),
        dpi=100, bbox_inches='tight')
    plt.close('all')

    # ── Integrated flux ───────────────────────────────────────────────
    flux_df = bayes.results.get_point_source_flux(10 * u.keV, 10 * u.MeV)
    log.info('  10 keV-10 MeV flux = %s', flux_df)

    # Extract the raw floats by stripping the astropy units (.value)
    f_val = flux_df['flux'].iloc[0].value
    f_low = flux_df['low bound'].iloc[0].value
    f_hi  = flux_df['hi bound'].iloc[0].value

    # ── Collect parameters ────────────────────────────────────────────
    df  = bayes.results.get_data_frame('equal tail')
    df  = df.drop('unit', axis=1, errors='ignore')
    param_array = df.to_numpy().flatten()
    
    full_row = np.append(param_array, [f_val, f_low, f_hi]) # Append the 3 flux values to the end of the parameter array
    fit_rows.append(full_row)
    log.info('  Bin %d / %d complete.', i + 1, n_bins)

print(f'\nAll {n_bins} bins fitted successfully.')


14:29:57  INFO      ==== Time bin 1 / 4 ====


 *****************************************************
 MultiNest v3.10
 Copyright Farhan Feroz & Mike Hobson
 Release Jul 2015

 no. of live points =  400
 dimensionality =    3
 *****************************************************
  analysing data from chains/fit-.txt
 ln(ev)=  -1261.1991511016140      +/-   3.0904763441954988E-002
 Total Likelihood Evaluations:         1499
 Sampling finished. Exiting MultiNest


Maximum a posteriori probability (MAP) point:

,result,unit
parameter,,
GBM.spectrum.main.Cutoff_powerlaw.K,(0.0010 +0.0022 +1.6) x 10^-2,1 / (keV s cm2)
GBM.spectrum.main.Cutoff_powerlaw.index,-1.19 -0.5 +0.28,
GBM.spectrum.main.Cutoff_powerlaw.xc,(3.4 +/- 1.9) x 10^2,keV


Values of -log(posterior) at the minimum:

...

processing Bayesian analyses:   0%|                                                                           …

Propagating errors:   0%|                                                                                    |…

processing Bayesian analyses:   0%|                                                                           …

14:32:46  INFO        10 keV-10 MeV flux =                                            flux  \
GBM: total  9.168197349416221e-10 erg / (s cm2)   

                                       low bound  \
GBM: total  1.0609708404571261e-11 erg / (s cm2)   

                                       hi bound  
GBM: total  7.134206887540033e-09 erg / (s cm2)  
14:32:46  INFO        Bin 4 / 4 complete.



All 4 bins fitted successfully.


## Cell 11 — Save Results to CSV

In [32]:
fit_array = np.array(fit_rows)
cutoffpl_fit_param = Table(fit_array, names=PARAM_NAMES)

# ── Attach bin edges as extra columns ────────────────────────────────
cutoffpl_fit_param['t_start'] = bb_starts
cutoffpl_fit_param['t_stop']  = bb_stops
cutoffpl_fit_param['dt']      = bb_stops - bb_starts

csv_path = os.path.join(MODEL_PATHS['root'], 'fit_params.csv')
cutoffpl_fit_param.write(csv_path, overwrite=True)

print(f'Results saved to: {csv_path}')
print(f'Shape: {fit_array.shape}  ({n_bins} bins x 16 parameters)')
cutoffpl_fit_param


Results saved to: gbm_pipeline_output/cutoffpl/fit_params.csv
Shape: (4, 15)  (4 bins x 16 parameters)


B_K,B_K_n_err,B_K_p_err,B_K_avg_err,B_alpha,B_alpha_n_err,B_alpha_p_err,B_alpha_avg_err,E_c,E_c_n_err,E_c_p_err,E_c_avg_err,flux,flux_low,flux_hi,t_start,t_stop,dt
float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
1.02233233795919e-05,2.2166765518049586e-05,0.01647436276483008,0.008248264765174065,-1.1867544338023452,-0.5312779989368559,0.28457496312591857,0.40792648103138723,341.1042456923748,-191.77618222857836,189.077223314683,190.4267027716307,6.786456516392241e-11,2.7401683262561387e-12,2.2640942325703064e-09,-9.999761998653412,-6.61941796541214,3.380344033241272
0.2063899777973935,0.1442923028221928,1.0955633588200444,0.6199278308211186,-0.8000538786059469,-0.585689619245131,-0.17809586565564983,0.20379687679474057,105.72599198788726,29.11183806498684,255.86033954942747,142.48608880720715,8.79451129170402e-08,7.630221041759745e-08,1.0338554767633803e-07,-6.61941796541214,7.057295024394989,13.676712989807129
0.8490736273585491,0.04964729950785962,1.6218306686193962,0.8357389840636279,-0.9962230972483954,-0.3357674401168298,-0.023461294435303426,0.15615307284076319,158.61964563878487,5.117434129076031,249.30737692599106,127.21240552753355,2.3051416300112852e-07,1.9977910000237018e-07,2.682372406153906e-07,7.057295024394989,11.157336056232452,4.100041031837463
1.0838759548643596e-05,9.067590740899736e-05,0.1689025394849446,0.0844966076961768,-1.168585659283913,-0.5832452562405905,0.20029313884679045,0.3917691975436905,353.727923825351,-232.44919772768338,150.362543064409,191.4058703960462,9.168197349416221e-10,1.0609708404571261e-11,7.134206887540033e-09,11.157336056232452,49.99896001815796,38.84162396192551


## Cell 12 — Pipeline Summary

In [33]:
print('=' * 60)
print('  GBM PIPELINE SUMMARY')
print('=' * 60)
print(f'  Trigger          : {CONFIG["trigger_name"]}')
print(f'  Detectors used   : {detectors}')
print(f'  Time bins fitted : {n_bins}')
print(f'  Bin edges (s)    : starts={bb_starts}, stops={bb_stops}')
print('-' * 60)
print('  COMMON OUTPUTS:')
print(f'  Base Directory   : {COMMON_PATHS["root"]}')
print(f'  Lightcurves      : {COMMON_PATHS["lc"]}')
print('-' * 60)
print('  MODEL SPECIFIC OUTPUTS:')

# Dynamically scan the root directory for model folders
found_models = False
if os.path.exists(COMMON_PATHS['root']):
    for item in sorted(os.listdir(COMMON_PATHS['root'])):
        model_dir = os.path.join(COMMON_PATHS['root'], item)
        
        # Check if it is a directory and not the shared 'lightcurves' folder
        if os.path.isdir(model_dir) and item != 'lightcurves':
            found_models = True
            print(f'  Model: [{item.upper()}]')
            print(f'    Spectra plots  : {os.path.join(model_dir, "spectra")}')
            print(f'    Corner plots   : {os.path.join(model_dir, "corners")}')
            
            # Check for the CSV file (adjust name if you used something else in Cell 11)
            csv_file = os.path.join(model_dir, 'fit_params.csv') 
            if os.path.exists(csv_file):
                print(f'    CSV results    : {csv_file}')
            else:
                print(f'    CSV results    : (Not generated)')
            print() # Blank line for spacing

if not found_models:
    print('  (No model-specific outputs found yet.)')

print('=' * 60)

  GBM PIPELINE SUMMARY
  Trigger          : bn140304557
  Detectors used   : ['n3', 'n4', 'b0']
  Time bins fitted : 4
  Bin edges (s)    : starts=[-9.999762   -6.61941797  7.05729502 11.15733606], stops=[-6.61941797  7.05729502 11.15733606 49.99896002]
------------------------------------------------------------
  COMMON OUTPUTS:
  Base Directory   : gbm_pipeline_output
  Lightcurves      : gbm_pipeline_output/lightcurves
------------------------------------------------------------
  MODEL SPECIFIC OUTPUTS:
  Model: [CUTOFFPL]
    Spectra plots  : gbm_pipeline_output/cutoffpl/spectra
    Corner plots   : gbm_pipeline_output/cutoffpl/corners
    CSV results    : gbm_pipeline_output/cutoffpl/fit_params.csv



## Cell 10 — Time-Resolved Spectral Fitting (Band Model)

In [34]:
MODEL_PATHS = get_model_paths('band')

PARAM_NAMES = [
    'B_K',     'B_K_n_err',     'B_K_p_err',     'B_K_avg_err',
    'B_alpha', 'B_alpha_n_err', 'B_alpha_p_err',  'B_alpha_avg_err',
    'E_peak',  'E_peak_n_err',  'E_peak_p_err',   'E_peak_avg_err',
    'beta',    'beta_n_err',    'beta_p_err',      'beta_avg_err',
    'flux',    'flux_low',      'flux_hi'
]

fit_rows = []

for i in range(n_bins):
    log.info('==== Time bin %d / %d ====', i + 1, n_bins)

    # ── Assemble per-detector plugins for this bin ────────────────────
    det_plugins = []
    for det in detectors:
        p = time_resolved_plugins[det][i]
        if det.startswith('b'):
            p.set_active_measurements(CONFIG['bgo_range'])
        else:
            ranges = [r.strip() for r in CONFIG['nai_range'].split(',')]
            p.set_active_measurements(*ranges)
        det_plugins.append(p)

    data_list = DataList(*det_plugins)

    # ── Fresh Band model per bin ──────────────────────────────────────
    model = build_band_model(CONFIG['ra'], CONFIG['dec'], CONFIG['priors_band'])

    # ── Bayesian fit with MultiNest ───────────────────────────────────
    bayes = BayesianAnalysis(model, data_list)
    bayes.set_sampler('multinest')
    bayes.sampler.setup(n_live_points=CONFIG['n_live_points'])
    bayes.sample()

    # ── νFν spectrum plot ─────────────────────────────────────────────
    fig_spec = plot_spectra(
        bayes.results,
        ene_min=1.0, ene_max=1e5, num_ene=200,
        contour_colors='white',
        contour_style_kwargs=dict(alpha=0.2),
        use_components=True, components_to_use=['total'],
        flux_unit='keV2/(cm2 s keV)',
    )
    ax = plt.gca()
    ax.set_xlabel('Energy (keV)', fontsize=16)
    ax.set_ylabel(r'$\nu\,F_\nu\;[\mathrm{keV\,cm^{-2}\,s^{-1}}]$', fontsize=16)
    ax.set_title(f"{CONFIG['trigger_name']}  bin {i+1}/{n_bins}  "
                 f"[{bb_starts[i]:.3f}, {bb_stops[i]:.3f}] s", fontsize=13)
    fig_spec.savefig(
        os.path.join(MODEL_PATHS['spectra'], f'band_spec_bin{i+1:03d}.png'),
        dpi=120, bbox_inches='tight')
    plt.close('all')

    # ── Corner plot ───────────────────────────────────────────────────
    fig_corner = bayes.results.corner_plot()
    fig_corner.savefig(
        os.path.join(MODEL_PATHS['corners'], f'band_corner_bin{i+1:03d}.png'),
        dpi=100, bbox_inches='tight')
    plt.close('all')

    # ── Integrated flux ───────────────────────────────────────────────
    flux_df = bayes.results.get_point_source_flux(10 * u.keV, 10 * u.MeV)
    log.info('  10 keV-10 MeV flux = %s', flux_df)

    # Extract the raw floats by stripping the astropy units (.value)
    f_val = flux_df['flux'].iloc[0].value
    f_low = flux_df['low bound'].iloc[0].value
    f_hi  = flux_df['hi bound'].iloc[0].value

    # ── Collect parameters ────────────────────────────────────────────
    df  = bayes.results.get_data_frame('equal tail')
    df  = df.drop('unit', axis=1, errors='ignore')
    param_array = df.to_numpy().flatten()
    
    full_row = np.append(param_array, [f_val, f_low, f_hi]) # Append the 3 flux values to the end of the parameter array
    fit_rows.append(full_row)
    log.info('  Bin %d / %d complete.', i + 1, n_bins)

print(f'\nAll {n_bins} bins fitted successfully.')


14:32:46  INFO      ==== Time bin 1 / 4 ====


 *****************************************************
 MultiNest v3.10
 Copyright Farhan Feroz & Mike Hobson
 Release Jul 2015

 no. of live points =  400
 dimensionality =    4
 *****************************************************
  analysing data from chains/fit-.txt ln(ev)=  -1263.0085179923631      +/-   6.6932896644991566E-002
 Total Likelihood Evaluations:         2593
 Sampling finished. Exiting MultiNest



Maximum a posteriori probability (MAP) point:

,result,unit
parameter,,
GBM.spectrum.main.Band.K,(1.0 +0.4 +4) x 10^-4,1 / (keV s cm2)
GBM.spectrum.main.Band.alpha,(-8 +/- 4) x 10^-1,
GBM.spectrum.main.Band.xp,(3.1 -1.6 +1.8) x 10^2,keV
GBM.spectrum.main.Band.beta,-2.5 -0.9 +0.4,


Values of -log(posterior) at the minimum:

...

processing Bayesian analyses:   0%|                                                                           …

Propagating errors:   0%|                                                                                    |…

processing Bayesian analyses:   0%|                                                                           …

14:36:16  INFO        10 keV-10 MeV flux =                                             flux  \
GBM: total  1.0261725745128875e-08 erg / (s cm2)   

                                      low bound  \
GBM: total  5.162358642298071e-09 erg / (s cm2)   

                                        hi bound  
GBM: total  1.7924495251950004e-08 erg / (s cm2)  
14:36:16  INFO        Bin 4 / 4 complete.



All 4 bins fitted successfully.


In [35]:
## Cell 11 — Save Results to CSV

In [36]:
fit_array = np.array(fit_rows)
band_fit_param = Table(fit_array, names=PARAM_NAMES)

# ── Attach bin edges as extra columns ────────────────────────────────
band_fit_param['t_start'] = bb_starts
band_fit_param['t_stop']  = bb_stops
band_fit_param['dt']      = bb_stops - bb_starts

csv_path = os.path.join(MODEL_PATHS['root'], 'fit_params.csv')
band_fit_param.write(csv_path, overwrite=True)

print(f'Results saved to: {csv_path}')
print(f'Shape: {fit_array.shape}  ({n_bins} bins x 16 parameters)')
band_fit_param


Results saved to: gbm_pipeline_output/band/fit_params.csv
Shape: (4, 19)  (4 bins x 16 parameters)


B_K,B_K_n_err,B_K_p_err,B_K_avg_err,B_alpha,B_alpha_n_err,B_alpha_p_err,B_alpha_avg_err,E_peak,E_peak_n_err,E_peak_p_err,E_peak_avg_err,beta,beta_n_err,beta_p_err,beta_avg_err,flux,flux_low,flux_hi,t_start,t_stop,dt
float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
0.00010291457110998725,3.891121412241734e-05,0.000382654845098712,0.00021078302961056468,-0.816619490061527,-0.3607308833580708,0.4415101347504717,0.4011205090542712,306.88735682812126,-161.98724339716145,183.454025419429,172.7206344082952,-2.5337944811285533,-0.8952479652654826,0.3924490029885259,0.6438484841270042,1.4376534214000162e-08,4.842226529475767e-09,3.502057579376095e-08,-9.999761998653412,-6.61941796541214,3.380344033241272
0.003134832445174829,-0.001061386520658438,0.0020948925991932064,0.0015781395599258222,-1.1471818286266033,-0.2219424522518032,0.3340997148126681,0.27802108353223565,156.54723033397804,-38.70563911159391,101.13853411622935,69.92208661391163,-2.6759824977107662,-0.796656955273189,0.3973837132470641,0.5970203342601266,1.0537345195877193e-07,8.587315016720202e-08,1.4214710798604347e-07,-6.61941796541214,7.057295024394989,13.676712989807129
0.008173263260483772,-0.003020090573347672,0.0028897515012717406,0.002954921037309706,-1.0529967245277116,-0.2653285098450975,0.20845881839258107,0.23689366411883928,162.64901249710593,-30.669709042718353,109.61434774892982,70.14202839582408,-2.6342098393539857,-0.8657137133759365,0.3383571617685486,0.6020354375722425,2.6727013181822476e-07,2.2151075872215716e-07,3.423257559030609e-07,7.057295024394989,11.157336056232452,4.100041031837463
0.00012219184733028497,1.3593717693341799e-05,0.000336864181477102,0.0001752289495852219,-1.1864228840349158,-0.2332798545615833,0.6257442807131502,0.42951206763736677,276.49128928387074,-189.82919468140858,111.12416292972637,150.47667880556747,-2.7512920149267424,-0.7183772128479045,0.529570832761658,0.6239740228047812,1.0261725745128875e-08,5.162358642298071e-09,1.7924495251950004e-08,11.157336056232452,49.99896001815796,38.84162396192551


## Cell 12 — Pipeline Summary

In [37]:
print('=' * 60)
print('  GBM PIPELINE SUMMARY')
print('=' * 60)
print(f'  Trigger          : {CONFIG["trigger_name"]}')
print(f'  Detectors used   : {detectors}')
print(f'  Time bins fitted : {n_bins}')
print(f'  Bin edges (s)    : starts={bb_starts}, stops={bb_stops}')
print('-' * 60)
print('  COMMON OUTPUTS:')
print(f'  Base Directory   : {COMMON_PATHS["root"]}')
print(f'  Lightcurves      : {COMMON_PATHS["lc"]}')
print('-' * 60)
print('  MODEL SPECIFIC OUTPUTS:')

# Dynamically scan the root directory for model folders
found_models = False
if os.path.exists(COMMON_PATHS['root']):
    for item in sorted(os.listdir(COMMON_PATHS['root'])):
        model_dir = os.path.join(COMMON_PATHS['root'], item)
        
        # Check if it is a directory and not the shared 'lightcurves' folder
        if os.path.isdir(model_dir) and item != 'lightcurves':
            found_models = True
            print(f'  Model: [{item.upper()}]')
            print(f'    Spectra plots  : {os.path.join(model_dir, "spectra")}')
            print(f'    Corner plots   : {os.path.join(model_dir, "corners")}')
            
            # Check for the CSV file (adjust name if you used something else in Cell 11)
            csv_file = os.path.join(model_dir, 'fit_params.csv') 
            if os.path.exists(csv_file):
                print(f'    CSV results    : {csv_file}')
            else:
                print(f'    CSV results    : (Not generated)')
            print() # Blank line for spacing

if not found_models:
    print('  (No model-specific outputs found yet.)')

print('=' * 60)

  GBM PIPELINE SUMMARY
  Trigger          : bn140304557
  Detectors used   : ['n3', 'n4', 'b0']
  Time bins fitted : 4
  Bin edges (s)    : starts=[-9.999762   -6.61941797  7.05729502 11.15733606], stops=[-6.61941797  7.05729502 11.15733606 49.99896002]
------------------------------------------------------------
  COMMON OUTPUTS:
  Base Directory   : gbm_pipeline_output
  Lightcurves      : gbm_pipeline_output/lightcurves
------------------------------------------------------------
  MODEL SPECIFIC OUTPUTS:
  Model: [BAND]
    Spectra plots  : gbm_pipeline_output/band/spectra
    Corner plots   : gbm_pipeline_output/band/corners
    CSV results    : gbm_pipeline_output/band/fit_params.csv

  Model: [CUTOFFPL]
    Spectra plots  : gbm_pipeline_output/cutoffpl/spectra
    Corner plots   : gbm_pipeline_output/cutoffpl/corners
    CSV results    : gbm_pipeline_output/cutoffpl/fit_params.csv

